In [ ]:
import csv
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.interpolate import interp1d

ELECTRON_DIR = "data_iron"
source_file = "source.txt"

# Defina as fontes que sua supervisora quer destacar
# Ordena os índices das fontes com base no t_acc em 1 GeV (do menor para o maior)
indices_ordenados = np.argsort(t_acc_1GeV_array)

# Pegando os índices das 3 fontes mais baixas (lower) e 3 mais altas (upper)
qtd_destaque = 3
indices_lower = indices_ordenados[:qtd_destaque]
indices_upper = indices_ordenados[-qtd_destaque:]

# Juntando tudo em uma única lista de índices
indices_destaque = np.concatenate((indices_lower, indices_upper))

# Criando a lista de nomes dinamicamente
fontes_destaque = [nomes_array[i] for i in indices_destaque]

print(f"Fontes Lower identificadas: {[nomes_array[i] for i in indices_lower]}")
print(f"Fontes Upper identificadas: {[nomes_array[i] for i in indices_upper]}")
# 1. Criar uma grade comum de Energia (ex: de 1e-3 a 1e5 GeV com 200 pontos)
E_comum = np.logspace(-3, 5, 200)

dados_populacao = {
    'nomes': [],
    't_acc_matrix': [],
    't_cool_matrix': [],
    'E_max': [],
    't_acc_1GeV': [],
    'frac_sync_Emax': []
}

with open(source_file, mode='r') as csvfile:
    reader = csv.DictReader(csvfile)
    
    for row in reader:
        nome = row['name']
        filename = f"{ELECTRON_DIR}/{nome}_electrons.csv"
        
        if not os.path.exists(filename):
            continue
            
        # Listas temporárias para ler o arquivo
        E_tmp, acc_tmp, cool_tmp, sync_tmp = [], [], [], []
        
        with open(filename, 'r') as f:
            csv_reader = csv.DictReader(f)
            for d in csv_reader:
                E_tmp.append(float(d['E_GeV']))
                acc_tmp.append(float(d['rate_acc']))
                
                # Soma das perdas totais
                total_cool = float(d['rate_sync']) + float(d['rate_bremss']) + float(d['rate_ic_0']) + float(d['rate_ic_1'])
                cool_tmp.append(total_cool)
                sync_tmp.append(float(d['rate_sync']))
                
        # Interpolação para a grade comum
        f_acc = interp1d(E_tmp, acc_tmp, fill_value="extrapolate")
        f_cool = interp1d(E_tmp, cool_tmp, fill_value="extrapolate")
        f_sync = interp1d(E_tmp, sync_tmp, fill_value="extrapolate")
        
        acc_interp = f_acc(E_comum)
        cool_interp = f_cool(E_comum)
        sync_interp = f_sync(E_comum)
        
        # Encontrar E_max (onde o resfriamento supera a aceleração)
        # Pegamos o primeiro índice onde cool > acc
        idx_max = np.where(cool_interp > acc_interp)[0]
        if len(idx_max) > 0:
            e_m = E_comum[idx_max[0]]
            frac_sync = sync_interp[idx_max[0]] / cool_interp[idx_max[0]]
        else:
            e_m = E_comum[-1] # Se não cruzar, assume o limite superior
            frac_sync = sync_interp[-1] / cool_interp[-1]
            
        # Taxa em 1 GeV (aproximadamente no índice onde E_comum é 1.0)
        idx_1gev = np.argmin(np.abs(E_comum - 1.0))
        t_acc_1g = acc_interp[idx_1gev]
        
        # Salvar na população
        dados_populacao['nomes'].append(nome)
        dados_populacao['t_acc_matrix'].append(acc_interp)
        dados_populacao['t_cool_matrix'].append(cool_interp)
        dados_populacao['E_max'].append(e_m)
        dados_populacao['t_acc_1GeV'].append(t_acc_1g)
        dados_populacao['frac_sync_Emax'].append(frac_sync)

# Converter matrizes para Numpy Arrays para facilitar os plots
t_acc_matrix = np.array(dados_populacao['t_acc_matrix'])
t_cool_matrix = np.array(dados_populacao['t_cool_matrix'])
E_max_array = np.array(dados_populacao['E_max'])
t_acc_1GeV_array = np.array(dados_populacao['t_acc_1GeV'])
nomes_array = np.array(dados_populacao['nomes'])

In [ ]:
plt.figure(figsize=(10, 7))
cmap = plt.get_cmap('viridis')
# Normaliza as cores pelo log do E_max
norm = plt.Normalize(vmin=np.log10(np.min(E_max_array)), vmax=np.log10(np.max(E_max_array)))

for i in range(len(nomes_array)):
    cor = cmap(norm(np.log10(E_max_array[i])))
    plt.loglog(E_comum, t_acc_matrix[i], color=cor, alpha=0.6, linewidth=1.5)

plt.xlabel('E [GeV]', fontsize=14)
plt.ylabel('$t_{acc}^{-1}$ [s$^{-1}$]', fontsize=14)
plt.xlim(1e-3, 1e4)
plt.ylim(1e0, 1e7)
plt.grid(True, which="both", ls="-", alpha=0.1)

# Adicionando a barra de cores
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm)
cbar.set_label('$\log_{10}$ E$_{max}$ [GeV]', fontsize=14)
plt.title('A. Mesmo espaguete, cor codificando $E_{max}$')
plt.show()

In [ ]:
# Ordenar as fontes pelo E_max (cria aquele visual de escadinha)
ordem = np.argsort(E_max_array)
matriz_ordenada_acc = t_acc_matrix[ordem]
matriz_ordenada_cool = t_cool_matrix[ordem]
E_max_ordenado = E_max_array[ordem]

# --- Para o Waterfall (Gráfico B) ---
plt.figure(figsize=(10, 7))
X, Y = np.meshgrid(E_comum, np.arange(len(nomes_array)))
Z_acc = np.log10(matriz_ordenada_acc)

plt.pcolormesh(X, Y, Z_acc, cmap='magma', shading='auto')
cbar = plt.colorbar()
cbar.set_label('$\log_{10}$ $t_{acc}^{-1}$ [s$^{-1}$]')

# Plotando a linha do E_max por cima
plt.plot(E_max_ordenado, np.arange(len(E_max_ordenado)), color='cyan', lw=2, label='$E_{max}$')

plt.xscale('log')
plt.xlim(1e-3, 1e4)
plt.xlabel('E [GeV]', fontsize=14)
plt.ylabel('Fonte (ordenada)', fontsize=14)
plt.title('B. Waterfall da populacao inteira')
plt.legend(loc='upper right')
plt.show()

# --- Para o Mapa Divergente (Gráfico C) ---
plt.figure(figsize=(10, 7))
# Razão entre aceleração e perda: valores > 0 acc vence, < 0 perda vence
Z_razao = np.log10(matriz_ordenada_acc / matriz_ordenada_cool)

plt.pcolormesh(X, Y, Z_razao, cmap='RdBu_r', shading='auto', vmin=-6, vmax=12)
cbar = plt.colorbar()
cbar.set_label('$\log_{10} (t_{acc}^{-1} / t_{perdas}^{-1})$')

plt.plot(E_max_ordenado, np.arange(len(E_max_ordenado)), color='black', lw=2, label='$E_{max}$')

plt.xscale('log')
plt.xlim(1e-3, 1e4)
plt.xlabel('E [GeV]', fontsize=14)
plt.ylabel('Fonte (ordenada)', fontsize=14)
plt.title('C. Onde a aceleracao vence')
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
frac = dados_populacao['frac_sync_Emax']

scatter = plt.scatter(t_acc_1GeV_array, E_max_array, c=frac, cmap='plasma', s=40)
cbar = plt.colorbar(scatter)
cbar.set_label('Fração Síncrotron no $E_{max}$', fontsize=14)

# Destacar fontes específicas
for i, nome in enumerate(nomes_array):
    if nome in fontes_destaque:
        plt.scatter(t_acc_1GeV_array[i], E_max_array[i], 
                    facecolors='none', edgecolors='black', s=200, lw=2)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('$t_{acc}^{-1}$ em 1 GeV [s$^{-1}$]', fontsize=14)
plt.ylabel('$E_{max}$ [GeV]', fontsize=14)
plt.title('E. Estrutura populacional com mecanismo dominante em cor')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

# Calcular percentis da amostra inteira (ao longo do eixo 0, que são as fontes)
p50 = np.percentile(t_acc_matrix, 50, axis=0) # Mediana
p16 = np.percentile(t_acc_matrix, 16, axis=0) # ~ 68% inferior
p84 = np.percentile(t_acc_matrix, 84, axis=0) # ~ 68% superior
p02 = np.percentile(t_acc_matrix, 2.5, axis=0) # ~ 95% inferior
p97 = np.percentile(t_acc_matrix, 97.5, axis=0) # ~ 95% superior

# Preencher bandas
plt.fill_between(E_comum, p02, p97, color='lightgray', label='95% da amostra')
plt.fill_between(E_comum, p16, p84, color='darkgray', label='68% da amostra')
plt.loglog(E_comum, p50, color='black', lw=2.5, label='Mediana')

# Plotar as fontes em destaque individualmente
cores = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']
idx_cor = 0
for i, nome in enumerate(nomes_array):
    if nome in fontes_destaque:
        plt.loglog(E_comum, t_acc_matrix[i], color=cores[idx_cor % len(cores)], lw=2, label=nome)
        idx_cor += 1

plt.xlabel('E [GeV]', fontsize=14)
plt.ylabel('$t_{acc}^{-1}$ [s$^{-1}$]', fontsize=14)
plt.xlim(1e-3, 1e14)
plt.ylim(1e-13, 1e5)
plt.legend(fontsize=9, loc='upper right', framealpha=1)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Fig 3: Histograma (Distribuição de t_acc em 1 GeV) ---
plt.figure(figsize=(8, 5))
log_t_acc = np.log10(t_acc_1GeV_array)

plt.hist(log_t_acc, bins=25, color='silver', edgecolor='gray')

idx_cor = 0
for i, nome in enumerate(nomes_array):
    if nome in fontes_destaque:
        plt.axvline(log_t_acc[i], color=cores[idx_cor % len(cores)], lw=2.5, label=nome)
        idx_cor += 1

plt.xlabel('$\log_{10}$ $t_{acc}^{-1}$ em E = 1 GeV [s$^{-1}$]', fontsize=14)
plt.ylabel('Número de fontes', fontsize=14)
plt.legend(fontsize=8)
plt.show()

# --- Fig 6: ECDF (Distribuição Acumulada de E_max) ---
plt.figure(figsize=(8, 5))
x = np.sort(E_max_array)
y = np.arange(1, len(x) + 1) / len(x) # Calcula a fração acumulada

plt.step(x, y, where='post', color='black', lw=2, alpha=0.7, label='Amostra Completa')

idx_cor = 0
for i, nome in enumerate(nomes_array):
    if nome in fontes_destaque:
        plt.axvline(E_max_array[i], color=cores[idx_cor % len(cores)], lw=2)
        idx_cor += 1

plt.xscale('log')
plt.xlabel('$E_{max}$ [GeV]', fontsize=14)
plt.ylabel('Fração acumulada de fontes', fontsize=14)
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.show()